# NetShield — Phase 4: Preprocessing & Feature Engineering

**Course:** CS-245 Machine Learning — Fall 2026  
**Project:** *Investigating Dataset-Dependent Behavior in Machine Learning-Based Network Intrusion Detection*

## Purpose

This notebook turns the findings from the EDA/comparison phase into a **leakage-safe preprocessing pipeline**.

The project is **not** trying to make the two datasets artificially identical. The goal is to:

1. clean genuine data-quality problems while preserving meaningful traffic behavior;
2. prepare native UNSW-NB15 and native CICIDS2017 representations for within-dataset experiments;
3. construct a carefully justified **common semantic representation** for cross-dataset experiments;
4. save auditable preprocessing decisions and diagnostics;
5. keep all fitted preprocessing decisions training-only.

> **Important:** This notebook does not train the final ML models. It prepares the data and artifacts needed for the next experiment notebook.


## Research principle

The novelty of NetShield is not "which classifier gets the highest score."

The central investigation is:

> **How and why does ML intrusion-detection performance change when the network dataset/environment changes, and can semantically aligned traffic features improve cross-dataset robustness?**

Therefore, preprocessing decisions are made with two constraints:

- remove invalid technical artifacts;
- do **not** erase genuine dataset-dependent differences merely because they make modeling harder.


In [1]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from pathlib import Path
import json
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

RANDOM_STATE = 42
TARGET = "label"

print("Configuration loaded.")


Configuration loaded.


## 2. Locate the NetShield project

Expected structure:

```text
NetShield/
├── data/
│   └── raw/
│       ├── UNSW-NB15/
│       └── CICIDS2017/
├── outputs/
└── notebooks/
```

Raw files are **read only**. This notebook never overwrites them.


In [2]:
# ============================================================
# 2. Project paths
# ============================================================

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "data" / "raw").exists():
            return p
    return Path.cwd()

PROJECT_ROOT = find_project_root()
RAW_ROOT = PROJECT_ROOT / "data" / "raw"
UNSW_DIR = RAW_ROOT / "UNSW-NB15"
CICIDS_DIR = RAW_ROOT / "CICIDS2017"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "preprocessing"
TABLE_DIR = OUTPUT_DIR / "tables"
DATA_DIR = OUTPUT_DIR / "prepared_data"
LOG_DIR = OUTPUT_DIR / "logs"

for folder in [OUTPUT_DIR, TABLE_DIR, DATA_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UNSW_DIR    :", UNSW_DIR)
print("CICIDS_DIR  :", CICIDS_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)


PROJECT_ROOT: c:\Users\BISMILLAH\Documents\NetShield
UNSW_DIR    : c:\Users\BISMILLAH\Documents\NetShield\data\raw\UNSW-NB15
CICIDS_DIR  : c:\Users\BISMILLAH\Documents\NetShield\data\raw\CICIDS2017
OUTPUT_DIR  : c:\Users\BISMILLAH\Documents\NetShield\outputs\preprocessing


## 3. Load the official UNSW train/test representation

We keep the provided UNSW training/testing split rather than creating a new random split.

This is important because the project is evaluating generalization, and the official train/test structure is itself part of the dataset definition.

The loader searches for the standard files:

- `UNSW_NB15_training-set.csv`
- `UNSW_NB15_testing-set.csv`

If they are absent, the notebook stops rather than silently inventing a different UNSW split.


In [3]:
# ============================================================
# 3. UNSW loader
# ============================================================

unsw_train_candidates = list(UNSW_DIR.rglob("UNSW_NB15_training-set.csv"))
unsw_test_candidates = list(UNSW_DIR.rglob("UNSW_NB15_testing-set.csv"))

if not unsw_train_candidates or not unsw_test_candidates:
    raise FileNotFoundError(
        "Standard UNSW_NB15_training-set.csv / UNSW_NB15_testing-set.csv "
        "were not found under data/raw/UNSW-NB15. "
        "Place the official train/test CSVs there before continuing."
    )

UNSW_TRAIN_PATH = unsw_train_candidates[0]
UNSW_TEST_PATH = unsw_test_candidates[0]

unsw_train_raw = pd.read_csv(UNSW_TRAIN_PATH)
unsw_test_raw = pd.read_csv(UNSW_TEST_PATH)

print("UNSW train:", unsw_train_raw.shape)
print("UNSW test :", unsw_test_raw.shape)
print("Train labels:", unsw_train_raw["label"].value_counts(dropna=False).to_dict())
print("Test labels :", unsw_test_raw["label"].value_counts(dropna=False).to_dict())


UNSW train: (175341, 45)
UNSW test : (82332, 45)
Train labels: {1: 119341, 0: 56000}
Test labels : {1: 45332, 0: 37000}


## 4. Load CICIDS2017

The eight CICIDS2017 machine-learning CSV files are concatenated, but the originating CSV is retained temporarily as **metadata**.

### Critical leakage rule

`source_file` is **not** a model feature.

Our EDA showed that particular source files are strongly associated with particular attack scenarios. It is therefore useful for:

- group-aware evaluation,
- diagnostics,
- error analysis,

but dangerous as a predictive input.


In [4]:
# ============================================================
# 4. CICIDS loader
# ============================================================

cicids_files = sorted(
    [p for p in CICIDS_DIR.rglob("*.csv") if p.is_file()]
)

if len(cicids_files) == 0:
    raise FileNotFoundError(
        "No CICIDS2017 CSV files were found under data/raw/CICIDS2017."
    )

print(f"CICIDS CSV files found: {len(cicids_files)}")
for p in cicids_files:
    print(" -", p.name)

cicids_parts = []

for path in cicids_files:
    df_part = pd.read_csv(path, low_memory=False)
    df_part["source_file"] = path.name
    cicids_parts.append(df_part)

cicids_raw = pd.concat(cicids_parts, ignore_index=True)

print("\nCICIDS shape:", cicids_raw.shape)
print("Label column candidates:",
      [c for c in cicids_raw.columns if str(c).strip().lower() == "label"])


CICIDS CSV files found: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv

CICIDS shape: (2830743, 80)
Label column candidates: [' Label']


## 5. Standardize column names

We normalize whitespace around column names only.

We do **not** rename semantic features into something that merely "looks similar" across datasets. Semantic alignment is handled explicitly later.


In [5]:
# ============================================================
# 5. Column-name cleanup
# ============================================================

def clean_column_names(df):
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out

unsw_train = clean_column_names(unsw_train_raw)
unsw_test = clean_column_names(unsw_test_raw)
cicids = clean_column_names(cicids_raw)

# Normalize CICIDS label spelling/capitalization only.
cicids_label_col = next(
    c for c in cicids.columns if str(c).strip().lower() == "label"
)
cicids = cicids.rename(columns={cicids_label_col: "Label"})

print("UNSW label:", "label" in unsw_train.columns)
print("CICIDS label:", "Label" in cicids.columns)


UNSW label: True
CICIDS label: True


# Part A — Native dataset preprocessing

## 6. Label normalization

For the main cross-dataset investigation, the primary target is **binary intrusion detection**:

- `0` = benign/normal
- `1` = attack

This is deliberately conservative because the attack taxonomies of UNSW-NB15 and CICIDS2017 are not directly equivalent.

Multiclass/attack-family analysis will be a later experiment with a separately justified taxonomy.


In [6]:
# ============================================================
# 6. Binary label normalization
# ============================================================

def normalize_unsw_binary_label(series):
    # UNSW 'label' is already binary in the supplied train/test representation.
    return pd.to_numeric(series, errors="coerce").astype("Int64")

def normalize_cicids_binary_label(series):
    s = series.astype(str).str.strip()
    return (s.str.upper() != "BENIGN").astype("int8")

unsw_train["label_binary"] = normalize_unsw_binary_label(unsw_train["label"])
unsw_test["label_binary"] = normalize_unsw_binary_label(unsw_test["label"])
cicids["label_binary"] = normalize_cicids_binary_label(cicids["Label"])

print("UNSW train binary:")
print(unsw_train["label_binary"].value_counts(dropna=False))

print("\nUNSW test binary:")
print(unsw_test["label_binary"].value_counts(dropna=False))

print("\nCICIDS binary:")
print(cicids["label_binary"].value_counts(dropna=False))


UNSW train binary:
label_binary
1    119341
0     56000
Name: count, dtype: Int64

UNSW test binary:
label_binary
1    45332
0    37000
Name: count, dtype: Int64

CICIDS binary:
label_binary
0    2273097
1     557646
Name: count, dtype: int64


## 7. Native feature exclusion rules

### UNSW

Exclude:

- `id` — identifier
- `label` — target
- `attack_cat` — target-derived attack category, not a predictive input

Potentially useful categorical traffic fields such as `proto`, `service`, and `state` are **not dropped merely because they are categorical**. They will be encoded later inside a model-specific pipeline.

### CICIDS

Exclude:

- `Label` — target
- `label_binary` — target
- `source_file` — metadata/group variable
- identifier/address/timestamp columns if present in the raw file (`Flow ID`, `Source IP`, `Destination IP`, `Timestamp`, etc.)

We do not use metadata to improve predictive performance.


In [7]:
# ============================================================
# 7. Define native feature sets
# ============================================================

UNSW_EXCLUDE = {
    "id",
    "label",
    "attack_cat",
    "label_binary",
}

CICIDS_METADATA_EXCLUDE = {
    "Label",
    "label_binary",
    "source_file",
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
}

unsw_native_features = [
    c for c in unsw_train.columns
    if c not in UNSW_EXCLUDE
]

cicids_native_features = [
    c for c in cicids.columns
    if c not in CICIDS_METADATA_EXCLUDE
]

print("UNSW native features:", len(unsw_native_features))
print("CICIDS native features:", len(cicids_native_features))


UNSW native features: 42
CICIDS native features: 78


## 8. Infinite values → missing values

CICIDS contains infinite values in rate-related fields.

We replace `+inf` and `-inf` with `NaN`.

We do **not** delete the affected rows automatically.

Why?

Because deleting those flows would make the dataset composition depend on a technical artifact. The safer first step is to mark the value as missing and let the training-only imputation stage handle it.


In [8]:
# ============================================================
# 8. Replace infinities with NaN
# ============================================================

def replace_inf_with_nan(df):
    out = df.copy()
    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out[numeric_cols].replace([np.inf, -np.inf], np.nan)
    return out

unsw_train = replace_inf_with_nan(unsw_train)
unsw_test = replace_inf_with_nan(unsw_test)
cicids = replace_inf_with_nan(cicids)

print("Infinite numeric cells after replacement:")
print("UNSW train:", np.isinf(unsw_train.select_dtypes(include=[np.number])).sum().sum())
print("UNSW test :", np.isinf(unsw_test.select_dtypes(include=[np.number])).sum().sum())
print("CICIDS    :", np.isinf(cicids.select_dtypes(include=[np.number])).sum().sum())


Infinite numeric cells after replacement:
UNSW train: 0
UNSW test : 0
CICIDS    : 0


## 9. Domain-invalid values

The EDA identified suspicious negative/overflow-like values in CICIDS.

We **do not blanket-replace every negative number** because some fields can legitimately use sentinel values such as `-1` (for example, initial TCP-window-related fields in some versions).

For this first preprocessing stage we therefore apply conservative rules only to features whose physical meaning clearly requires non-negative values:

- duration
- packet/byte counts
- byte/packet rates
- inter-arrival times
- packet/header lengths
- active/idle times

Suspicious values become `NaN`, not deleted rows.

`Init_Win_bytes_forward` and `Init_Win_bytes_backward` are intentionally excluded from this automatic rule and will be handled as a documented sentinel-sensitive feature in the native model pipeline.


In [9]:
# ============================================================
# 9. Conservative domain validation
# ============================================================

NONNEGATIVE_CICIDS_PATTERNS = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Flow IAT",
    "Fwd IAT",
    "Bwd IAT",
    "Fwd Packet Length",
    "Bwd Packet Length",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length",
    "Header Length",
    "Subflow Fwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Bwd Packets",
    "Subflow Bwd Bytes",
    "act_data_pkt_fwd",
    "min_seg_size_forward",
    "Active ",
    "Idle ",
]

def apply_cicids_domain_rules(df):
    out = df.copy()
    affected = []

    for col in out.select_dtypes(include=[np.number]).columns:
        if col in {"Init_Win_bytes_forward", "Init_Win_bytes_backward"}:
            continue

        if any(pattern in col for pattern in NONNEGATIVE_CICIDS_PATTERNS):
            mask = out[col] < 0
            n = int(mask.sum())
            if n:
                out.loc[mask, col] = np.nan
                affected.append({
                    "feature": col,
                    "invalid_negative_values_replaced": n
                })

    return out, pd.DataFrame(affected)

cicids, cicids_invalid_report = apply_cicids_domain_rules(cicids)

print(cicids_invalid_report if not cicids_invalid_report.empty else "No negative values matched the conservative rules.")


                 feature  invalid_negative_values_replaced
0          Flow Duration                               115
1           Flow Bytes/s                                85
2         Flow Packets/s                               115
3          Flow IAT Mean                               115
4           Flow IAT Max                               115
5           Flow IAT Min                              2891
6            Fwd IAT Min                                17
7      Fwd Header Length                                35
8      Bwd Header Length                                22
9    Fwd Header Length.1                                35
10  min_seg_size_forward                                35


## 10. Constant and near-constant features

CICIDS contains constant-zero features such as the bulk-statistics fields.

These do not contain predictive information and are safe to remove from the native CICIDS representation.

Near-constant features are **not automatically removed** at this stage. They may still carry a small amount of information, and automatic removal would turn an exploratory observation into an unsupported modeling assumption.

A later train-only feature-selection sensitivity analysis can test whether removing them changes conclusions.


In [10]:
# ============================================================
# 10. Remove CICIDS constant features
# ============================================================

def find_constant_columns(df, feature_cols):
    constants = []
    for c in feature_cols:
        if c in df.columns and df[c].nunique(dropna=False) <= 1:
            constants.append(c)
    return constants

cicids_constant_features = find_constant_columns(cicids, cicids_native_features)

print("Constant CICIDS features:")
for c in cicids_constant_features:
    print(" -", c)

cicids_native_features_clean = [
    c for c in cicids_native_features
    if c not in cicids_constant_features
]

print("\nCICIDS native features before:", len(cicids_native_features))
print("CICIDS native features after :", len(cicids_native_features_clean))


Constant CICIDS features:
 - Bwd PSH Flags
 - Bwd URG Flags
 - Fwd Avg Bytes/Bulk
 - Fwd Avg Packets/Bulk
 - Fwd Avg Bulk Rate
 - Bwd Avg Bytes/Bulk
 - Bwd Avg Packets/Bulk
 - Bwd Avg Bulk Rate

CICIDS native features before: 78
CICIDS native features after : 70


# Part B — Duplicate/leakage diagnostics

## 11. Duplicate analysis before deciding what to remove

The EDA found a large number of duplicates in CICIDS.

We will **not blindly delete them**.

Instead, we inspect whether duplicate feature rows:

1. have conflicting labels;
2. appear across source files;
3. could cross train/test boundaries.

This matters because duplicate traffic observations appearing in both training and testing can inflate performance.

The final experiment notebook will use a duplicate-aware split strategy.


In [11]:
# ============================================================
# 11. Duplicate conflict diagnostics
# ============================================================

# Work on the cleaned native CICIDS feature space.
dup_check_cols = cicids_native_features_clean + ["label_binary"]

# Exact duplicate rows across model features + label.
exact_duplicate_count = int(
    cicids.duplicated(subset=dup_check_cols, keep=False).sum()
)

# Same feature vector with more than one binary label.
feature_only_cols = cicids_native_features_clean
label_conflicts = (
    cicids.groupby(feature_only_cols, dropna=False)["label_binary"]
    .nunique()
    .reset_index(name="n_labels")
)

conflicting_feature_groups = int((label_conflicts["n_labels"] > 1).sum())

duplicate_report = pd.DataFrame({
    "metric": [
        "duplicate rows on native features + label",
        "feature groups with conflicting binary labels",
    ],
    "value": [
        exact_duplicate_count,
        conflicting_feature_groups,
    ]
})

duplicate_report.to_csv(
    TABLE_DIR / "duplicate_diagnostics.csv",
    index=False
)

duplicate_report


,metric,value
0,duplicate rows on native features + label,403550
1,feature groups with conflicting binary labels,698


# Part C — Common semantic representation

## 12. Why we are NOT using name matching

The datasets do not have an exact shared predictive schema.

Some superficially similar names are **not semantically equivalent**.

For example:

- UNSW `sload` / `dload` describe **bits per second**;
- CICIDS `Fwd Packets/s` / `Bwd Packets/s` describe **packets per second**.

Therefore we will **not** map them directly.

Instead, when possible, we derive a common quantity from primitive traffic measurements.

This is a key methodological choice in NetShield.


## 13. Common semantic features used in the first cross-dataset experiment

We will construct these quantities:

| Common feature | UNSW source/derivation | CICIDS source/derivation |
|---|---|---|
| `flow_duration_sec` | `dur` | `Flow Duration / 1,000,000` |
| `fwd_packets` | `spkts` | `Total Fwd Packets` |
| `bwd_packets` | `dpkts` | `Total Backward Packets` |
| `fwd_bytes` | `sbytes` | `Total Length of Fwd Packets` |
| `bwd_bytes` | `dbytes` | `Total Length of Bwd Packets` |
| `total_packets` | `spkts + dpkts` | `Total Fwd Packets + Total Backward Packets` |
| `total_bytes` | `sbytes + dbytes` | `Total Length of Fwd Packets + Total Length of Bwd Packets` |
| `packet_rate` | `total_packets / duration_sec` | `Flow Packets/s` |
| `fwd_packet_rate` | `spkts / duration_sec` | `Fwd Packets/s` |
| `bwd_packet_rate` | `dpkts / duration_sec` | `Bwd Packets/s` |
| `avg_packet_size` | `total_bytes / total_packets` | `Average Packet Size` |

### Unit alignment

CICIDS flow duration is reported in microseconds, so it is converted to seconds.

This avoids comparing the UNSW duration values against CICIDS microseconds directly.


In [13]:
# ============================================================
# 13. Build common semantic representation
# ============================================================

def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")

    # Convert scalar denominators (e.g., 1_000_000) into a Series
    if np.isscalar(denominator):
        denominator = pd.Series(denominator, index=numerator.index)
    else:
        denominator = pd.to_numeric(denominator, errors="coerce")

    # Avoid division by zero
    denominator = denominator.replace(0, np.nan)

    return numerator.div(denominator)

def build_unsw_common(df):
    out = pd.DataFrame(index=df.index)

    out["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce")
    out["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce")
    out["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce")
    out["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce")
    out["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce")

    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]

    out["packet_rate"] = safe_divide(
        out["total_packets"], out["flow_duration_sec"]
    )
    out["fwd_packet_rate"] = safe_divide(
        out["fwd_packets"], out["flow_duration_sec"]
    )
    out["bwd_packet_rate"] = safe_divide(
        out["bwd_packets"], out["flow_duration_sec"]
    )
    out["avg_packet_size"] = safe_divide(
        out["total_bytes"], out["total_packets"]
    )

    out["label_binary"] = df["label_binary"].astype("int8")
    return out

def build_cicids_common(df):
    out = pd.DataFrame(index=df.index)

    # CICIDS Flow Duration is in microseconds in the standard feature definition.
    out["flow_duration_sec"] = safe_divide(
        df["Flow Duration"], 1_000_000
    )
    out["fwd_packets"] = pd.to_numeric(
        df["Total Fwd Packets"], errors="coerce"
    )
    out["bwd_packets"] = pd.to_numeric(
        df["Total Backward Packets"], errors="coerce"
    )
    out["fwd_bytes"] = pd.to_numeric(
        df["Total Length of Fwd Packets"], errors="coerce"
    )
    out["bwd_bytes"] = pd.to_numeric(
        df["Total Length of Bwd Packets"], errors="coerce"
    )

    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]

    # Use the dataset's direct flow-rate field for total packet rate.
    out["packet_rate"] = pd.to_numeric(
        df["Flow Packets/s"], errors="coerce"
    )
    out["fwd_packet_rate"] = pd.to_numeric(
        df["Fwd Packets/s"], errors="coerce"
    )
    out["bwd_packet_rate"] = pd.to_numeric(
        df["Bwd Packets/s"], errors="coerce"
    )
    out["avg_packet_size"] = pd.to_numeric(
        df["Average Packet Size"], errors="coerce"
    )

    out["label_binary"] = df["label_binary"].astype("int8")
    return out

unsw_common_train = build_unsw_common(unsw_train)
unsw_common_test = build_unsw_common(unsw_test)
cicids_common = build_cicids_common(cicids)

print("UNSW common:", unsw_common_train.shape)
print("CICIDS common:", cicids_common.shape)


UNSW common: (175341, 12)
CICIDS common: (2830743, 12)


## 14. Common-feature validation

Before using the semantic representation in ML, we verify:

- required source columns exist;
- no accidental target leakage;
- units are aligned;
- impossible negative values are absent;
- infinite values are converted to missing;
- derived rates are not silently treated as raw equivalents.

This validation is part of the contribution: the common representation is based on **traffic semantics**, not column-name similarity.


In [14]:
# ============================================================
# 14. Validate common representation
# ============================================================

common_features = [
    "flow_duration_sec",
    "fwd_packets",
    "bwd_packets",
    "fwd_bytes",
    "bwd_bytes",
    "total_packets",
    "total_bytes",
    "packet_rate",
    "fwd_packet_rate",
    "bwd_packet_rate",
    "avg_packet_size",
]

def common_quality_report(df, name):
    rows = []
    for c in common_features:
        s = pd.to_numeric(df[c], errors="coerce")
        rows.append({
            "dataset": name,
            "feature": c,
            "missing": int(s.isna().sum()),
            "infinite": int(np.isinf(s).sum()),
            "negative": int((s < 0).sum()),
            "zero_rate": float((s == 0).mean()),
        })
    return pd.DataFrame(rows)

common_quality = pd.concat([
    common_quality_report(unsw_common_train, "UNSW_train"),
    common_quality_report(unsw_common_test, "UNSW_test"),
    common_quality_report(cicids_common, "CICIDS"),
], ignore_index=True)

common_quality.to_csv(
    TABLE_DIR / "common_feature_quality.csv",
    index=False
)

common_quality


,dataset,feature,missing,infinite,negative,zero_rate
0,UNSW_train,flow_duration_sec,0,0,0,0.01515
1,UNSW_train,fwd_packets,0,0,0,0.00000
2,UNSW_train,bwd_packets,0,0,0,0.48067
3,UNSW_train,fwd_bytes,0,0,0,0.00000
4,UNSW_train,bwd_bytes,0,0,0,0.48067
5,UNSW_train,total_packets,0,0,0,0.00000
6,UNSW_train,total_bytes,0,0,0,0.00000
7,UNSW_train,packet_rate,2657,0,0,0.00000
8,UNSW_train,fwd_packet_rate,2657,0,0,0.00000
9,UNSW_train,bwd_packet_rate,2657,0,0,0.46552


# Part D — Training-only preprocessing strategy

## 15. What will be fitted only on training data?

The following operations must be fitted using training data only:

- numeric imputation values;
- categorical encoding mappings;
- scaling parameters;
- feature-selection thresholds;
- model hyperparameter tuning;
- any learned transformation.

The test set is never used to calculate these quantities.

This follows the project's original leakage-control requirement.


In [15]:
# ============================================================
# 15. Training-only median imputation artifact for common features
# ============================================================

# This is an artifact demonstration for the common semantic representation.
# The final model notebook will use sklearn Pipelines so the same rule is
# enforced automatically during fitting.

common_train_numeric = unsw_common_train[common_features].copy()
common_train_medians = common_train_numeric.median()

common_train_medians.to_csv(
    TABLE_DIR / "unsw_common_training_medians.csv",
    header=["median"]
)

print("Training-only UNSW common-feature medians saved.")
common_train_medians


Training-only UNSW common-feature medians saved.


flow_duration_sec       0.00158
fwd_packets             2.00000
bwd_packets             2.00000
fwd_bytes             430.00000
bwd_bytes             164.00000
total_packets           4.00000
total_bytes           802.00000
packet_rate         4,192.87212
fwd_packet_rate     2,058.67216
bwd_packet_rate         4.80033
avg_packet_size        91.50000
dtype: float64

## 16. Prepared-data policy

We will **not** overwrite raw data.

For reproducibility, this notebook saves:

- feature inventories;
- duplicate diagnostics;
- common-feature quality diagnostics;
- semantic feature definitions;
- preprocessing decisions.

The next modeling notebook will create model-ready matrices using `sklearn` pipelines so imputation, encoding, and scaling are fitted on the training partition only.

This keeps preprocessing auditable and prevents accidental leakage.


In [16]:
# ============================================================
# 16. Save preprocessing specification
# ============================================================

preprocessing_spec = {
    "project": "NetShield",
    "target": "binary intrusion detection",
    "random_state": RANDOM_STATE,
    "native_unsw": {
        "keep_official_train_test_split": True,
        "exclude": sorted(UNSW_EXCLUDE),
        "fit_preprocessing_on_training_only": True,
    },
    "native_cicids": {
        "exclude_metadata": sorted(CICIDS_METADATA_EXCLUDE),
        "remove_constant_features": sorted(cicids_constant_features),
        "source_file_is_metadata_only": True,
        "fit_preprocessing_on_training_only": True,
    },
    "data_quality": {
        "replace_infinity_with_nan": True,
        "conservative_domain_invalid_values_to_nan": True,
        "do_not_blanket_delete_rows": True,
        "preserve_meaningful_zero_values": True,
        "do_not_automatically_remove_outliers": True,
        "do_not_automatically_remove_all_correlated_features": True,
    },
    "common_semantic_features": common_features,
    "semantic_alignment": {
        "duration_unit_alignment": "CICIDS microseconds converted to seconds",
        "rate_alignment": "derived/direct packet-rate quantities, not UNSW sload/dload",
        "name_matching_alone_is_not_accepted": True,
    },
    "future_modeling": {
        "primary_task": "binary classification",
        "main_metrics": [
            "precision",
            "recall",
            "f1",
            "macro_f1",
            "pr_auc",
            "false_positive_rate",
            "confusion_matrix",
        ],
        "class_weighting_preferred_before_smote": True,
    }
}

with open(LOG_DIR / "preprocessing_spec.json", "w", encoding="utf-8") as f:
    json.dump(preprocessing_spec, f, indent=2)

print("Saved:", LOG_DIR / "preprocessing_spec.json")


Saved: c:\Users\BISMILLAH\Documents\NetShield\outputs\preprocessing\logs\preprocessing_spec.json


# 17. Final decision summary

### Confirmed preprocessing decisions

| Decision | Status |
|---|---|
| Keep UNSW official train/test split | **Confirmed** |
| Binary target for primary transfer experiment | **Confirmed** |
| Replace ±inf with NaN | **Confirmed** |
| Preserve meaningful zeros | **Confirmed** |
| Do not blindly delete outliers | **Confirmed** |
| Remove CICIDS constant features | **Confirmed** |
| Exclude CICIDS `source_file` from predictors | **Confirmed** |
| Keep IDs/addresses/timestamps out of predictors | **Confirmed** |
| Training-only imputation/encoding/scaling | **Confirmed** |
| Do not automatically use SMOTE | **Confirmed** |
| Do not map UNSW `sload/dload` to CICIDS packet rates | **Confirmed** |
| Align CICIDS duration units to seconds | **Confirmed** |
| Build common rates from semantically matching quantities | **Confirmed** |

### Still evaluated in the modeling phase

- baseline random/stratified CICIDS split versus source-aware robustness split;
- duplicate-group leakage prevention;
- native-model feature selection sensitivity;
- attack-family multiclass taxonomy;
- whether semantic alignment actually improves cross-dataset performance.

These are kept as **experiments**, not silently assumed conclusions.


# Next notebook

## `05_modeling_experiments.ipynb`

The next stage will compare:

1. **UNSW → UNSW** native baseline
2. **CICIDS → CICIDS** native baseline
3. **UNSW → CICIDS** common semantic transfer
4. **CICIDS → UNSW** common semantic transfer

Then we will add the novelty-focused analyses:

- within-vs-cross performance gap;
- distribution-shift versus performance-drop analysis;
- class-wise transfer errors;
- confusion matrices;
- feature importance;
- robustness to alternative CICIDS split strategies.

The important point is that the models are now being used to **test the NetShield research hypotheses**, rather than simply ranking classifiers.
